# 校則なんでも相談ボット

新入生の質問に、校則をもとに答えるボットを作ります。

答えを作るのは [LLM-jp](https://llm-jp.nii.ac.jp/) の日本語 AI です。国立情報学研究所などが進めている、日本語のデータで学習された AI を使います。

画面の表示には [ui-hiroba](https://pypi.org/project/ui-hiroba/) を使います。サーバーを立てずに、セルの出力にそのままカードや表を表示できます。

## このノートブックのしくみ

質問がきたら、次の2段階で答えます。

1. 質問に近い条文を、校則の中からさがす
2. 見つけた条文だけを AI に渡して、やさしい言葉に直してもらう

AI に校則ぜんぶを渡すのではなく、関係のある条文だけを渡すのがポイントです。この作り方を RAG（検索してから答える方式）と呼びます。

> このノートブックは Google Colab で動かしてください。AI の計算に PyTorch を使うため、ブラウザ内で動く PyHiroba では実行できません。

In [ ]:
%pip install -q ui-hiroba

In [ ]:
import torch, transformers
import ui_hiroba as ui
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

MODEL_NAME = "llm-jp/llm-jp-3-980m-instruct3"
# 下の行を書きかえると、大きいAIと答えを比べられます
#   llm-jp/llm-jp-3-150m-instruct3 … 1.5億　約0.3GB
#   llm-jp/llm-jp-3-440m-instruct3 … 4.4億　約0.9GB
#   llm-jp/llm-jp-3-980m-instruct3 … 9.9億　約2.0GB

ui.alert(f"使うAI：{MODEL_NAME}", kind="info", title="準備")

## 1. 校則を用意する

校則を Python のリストにします。`ALIASES` は、条文には出てこないけれど生徒が使いそうな言葉です。条文をさがすときだけ使います。

In [ ]:
# 学校の校則　※教材用に作成した架空の規程です
SCHOOL_RULES = [
("第3条（登校時刻）", "朝は8時30分までに登校します。おくれた場合は遅刻になります。電車がおくれたときは、遅延証明書を出せば遅刻になりません。"),
("第4条（欠席の届出）", "休むとき、おくれるとき、早く帰るときは、8時15分までに保護者から学校に連絡します。3日以上続けて休むときは、医師の診断書が必要です。"),
("第5条（服装）", "登下校と校内では制服を着ます。体育の授業では体操服を着ます。制服を短くするなど、作りかえることはできません。"),
("第6条（衣替えと防寒具）", "夏服への切りかえは6月1日から15日まで、冬服への切りかえは10月1日から15日までです。冬はコートやマフラーを使えます。ただし校舎の中ではコートをぬぎます。"),
("第7条（頭髪と装飾品）", "髪は清潔にします。髪を染めることとパーマはできません。生まれつきの髪の色は、入学のときに担任に伝えます。化粧とピアスはできません。"),
("第8条（持ち物）", "勉強に必要のない物は持ってきません。多くのお金や高い物は持ってきません。なくしたり盗まれたりしたときは、すぐに担任に伝えます。"),
("第9条（電子機器）", "スマートフォンや携帯電話を学校に持ってくることはできます。ただし授業のある時間は、電源を切って鞄にしまいます。先生が使ってよいと言ったときは使えます。人を撮るときは、その人の許可をとります。"),
("第10条（通学の方法）", "歩くか電車やバスで通学します。自動車やバイクでの通学はできません。バイクや車の免許を取ることも、在学中はできません。"),
("第11条（自転車通学）", "自転車で通学するには、申し込んで学校の許可をもらいます。許可証を自転車にはります。ヘルメットをかぶり、保険に入ります。自転車は駐輪場にとめて鍵をかけます。"),
("第12条（アルバイト）", "アルバイトは原則できません。家庭の事情があるときは、保護者の同意を得て申し込み、学校の許可をもらえばできます。夜10時から朝5時までの仕事はできません。"),
("第13条（外泊と旅行）", "生徒だけで外泊したり泊まりがけで旅行したりするときは、保護者の同意を得て、前もって担任に届け出ます。長い休みの間も同じです。"),
("第14条（インターネット）", "インターネットで発信するとき、人を傷つけたり学校の名誉を傷つけたりしてはいけません。校内で撮った写真や動画を、写っている人の許可なくネットに出してはいけません。"),
("第15条（部活動）", "部活動に入るか入らないかは自由です。平日は午後6時までです。日曜日と祝日は原則として活動しません。定期テストの1週間前から、テストが終わるまで活動はありません。"),
("第16条（きまりを守れないとき）", "きまりを守れないときは、担任や先生が指導します。それでも直らないときは、保護者を交えて話し合います。"),
]

# 条文の言葉と、生徒が使う言葉をつなぐための言い換え語（条文をさがすときにだけ使います）
ALIASES = {
"第3条": "遅刻 ちこく 何時 朝 電車 バス 間に合わない",
"第4条": "休む 休んだら 欠席 やすむ 風邪 体調不良 早退 連絡 保護者 何日 続けて 診断書",
"第5条": "制服 服装 私服 ボタン 着る 体育",
"第6条": "コート マフラー 手袋 上着 寒い 暑い 衣替え 夏服 冬服",
"第7条": "髪 かみ 髪型 染める 色 パーマ ピアス 化粧 メイク 地毛",
"第8条": "持ち物 財布 お金 現金 貴重品 なくした 盗まれた ゲーム",
"第9条": "スマホ スマートフォン 携帯 ケータイ 電話 充電 イヤホン 音楽 撮影 写真 動画 授業中",
"第10条": "通学 徒歩 電車 バス バイク 原付 車 免許",
"第11条": "自転車 チャリ 駐輪場 ヘルメット 保険 許可証 鍵",
"第12条": "アルバイト バイト 働く 仕事 お金を稼ぐ 深夜",
"第13条": "外泊 旅行 泊まり 友達の家 長期休み 夏休み",
"第14条": "SNS インスタ X ツイッター 投稿 ネット 写真 動画 悪口",
"第15条": "部活 部活動 練習 日曜 祝日 テスト前 下校時刻 何時まで 活動",
"第16条": "違反 破ったら 守れない 指導 呼び出し 反省",
}

EXAMPLES = [
"スマホって学校に持っていっていいの？",
"アルバイトはできますか？",
"自転車で通学したいです",
"髪を染めてもいい？",
"部活は日曜日もある？",
"文化祭でクラスTシャツを作ってもいい？",
]

ui.table([{"条": t, "内容": b} for t, b in SCHOOL_RULES], caption="この校則ぜんぶ")

## 2. 質問に近い条文をさがす

文章どうしの似ている度合いを計算して、いちばん近い条文を選びます。ここは AI を使いません。

In [ ]:
search_texts = [f"{title} {body} {ALIASES.get(title.split('（')[0], '')}"
                for title, body in SCHOOL_RULES]
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 2), sublinear_tf=True)
rule_vectors = vectorizer.fit_transform(search_texts)

# 質問にいちばん近い条文と、その近さ（0〜1）を返します
def find_rule(question):
  scores = cosine_similarity(vectorizer.transform([question]), rule_vectors)[0]
  index = int(scores.argmax())
  title, body = SCHOOL_RULES[index]
  return title, body, float(scores[index])

# 例の質問がどの条文に結びつくか、AIを読み込む前に確かめられます
rows = []
for q in EXAMPLES:
  title, _, score = find_rule(q)
  rows.append({"質問": q, "見つかった条文": title, "近さ": f"{score:.2f}"})

ui.table(rows, caption="質問と条文の結びつき")

最後の「文化祭でクラスTシャツを作ってもいい？」だけ、近さの数字がとても小さくなっています。この校則には当てはまる条文がないからです。

このあと作るボットでは、近さが小さいときに注意を出すようにします。

## 3. AI を読み込む

ここは少し時間がかかります。1回だけ実行してください。

In [ ]:
use_gpu = torch.cuda.is_available()
dtype = torch.float16 if use_gpu else torch.float32
dtype_key = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
  MODEL_NAME, device_map="auto", **{dtype_key: dtype})
model.eval()

ui.alert("準備できました！", kind="success",
         title="GPUを使います" if use_gpu else "CPUを使います")

## 4. AI に渡す文をつくる

答え方のお手本を1組見せてから、本当の質問を渡します。こうすると、答えの調子がそろいます。

In [ ]:
SHOT_RULE = "スマートフォンや携帯電話を学校に持ってくることはできます。ただし授業のある時間は、電源を切って鞄にしまいます。"
SHOT_QUESTION = "スマホは持っていっていいですか。"
SHOT_ANSWER = "はい、持ってきていいです。ただし授業の時間は電源を切って鞄にしまってください。"
HEAD = "校則をもとに、新入生の質問にやさしい言葉で答えてください。\n"

def build_messages(body, question):
  return [
    {"role": "user", "content": f"{HEAD}校則：{SHOT_RULE}\n質問：{SHOT_QUESTION}"},
    {"role": "assistant", "content": SHOT_ANSWER},
    {"role": "user", "content": f"校則：{body}\n質問：{question}"},
  ]

# transformersのバージョンによって戻り値の形がちがうため、両方に対応します
def build_inputs(messages):
  encoded = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt")
  inputs = {"input_ids": encoded} if hasattr(encoded, "shape") else dict(encoded)
  return {name: value.to(model.device) for name, value in inputs.items()}

# 答えたあとにお手本の続きを書き始めることがあるため、そこで切ります
def trim(reply):
  for stop in ["校則：", "質問：", "答え：", "###"]:
    if stop in reply:
      reply = reply.split(stop)[0]
  return reply.strip()

print("準備できました")

## 5. 質問に答える

答えと、AI が読んだ条文を、カードにして表示します。近さが 0.15 より小さいときは、当てはまる条文がない可能性を知らせます。

In [ ]:
# 条文が見つかったといえる近さの下限。数字を変えると注意の出やすさが変わります
MIN_SCORE = 0.15

def ask(question):
  title, body, score = find_rule(question)

  inputs = build_inputs(build_messages(body, question))
  prompt_length = inputs["input_ids"].shape[-1]

  with torch.no_grad():
    generated = model.generate(
      **inputs, max_new_tokens=100, do_sample=False,
      repetition_penalty=1.15, no_repeat_ngram_size=3,
      pad_token_id=tokenizer.eos_token_id)

  reply = trim(tokenizer.decode(generated[0][prompt_length:], skip_special_tokens=True))
  if not reply:
    reply = "うまく答えられませんでした。質問の書き方を変えてみてください。"

  parts = [ui.card(question, reply)]
  if score < MIN_SCORE:
    parts.append(ui.alert(
      "この質問に近い条文が校則の中に見つかりませんでした。答えはAIの推測かもしれません。",
      kind="warning", title="気をつけて読んでください"))
  parts.append(ui.columns(
    ui.reveal(body, summary=f"AIが読んだ条文：{title}"),
    ui.stat("渡した文の長さ", prompt_length, unit="トークン"),
    widths=[3, 1],
  ))
  return ui.stack(*parts)

# 質問の文を書きかえて、何度でも実行してみましょう
ask("スマホって学校に持っていっていいの？")

## 6. 続けて質問する

入力欄に質問を打つと、続けて聞けます。終わるときは、何も入力せずに Enter を押してください。

In [ ]:
while True:
  question = input("質問（終わるときは、何も入力せずEnter）： ")
  if not question.strip():
    ui.show(ui.alert("おしまいです。おつかれさま！", kind="success"))
    break
  ui.show(ask(question))

## 7. 例の質問をまとめて試す

6つの例をまとめて実行して、答えを見比べます。

In [ ]:
for question in EXAMPLES:
  ui.show(ask(question))

## やってみよう

1. `MODEL_NAME` を小さいAI（`llm-jp-3-150m-instruct3`）に変えて、答えがどう変わるか比べてみましょう
2. `SCHOOL_RULES` に自分の学校の校則を1つ足して、その条文について質問してみましょう
3. `ALIASES` に言葉を足すと、条文が見つかりやすくなります。うまく見つからなかった質問で試してみましょう
4. `MIN_SCORE` の数字を大きくすると、注意が出やすくなります。いくつがちょうどよいか探してみましょう

## 考えてみよう

- AI は校則に書いていないことも、それらしく答えてしまうことがあります。どうすれば気づけるでしょうか
- 「AIが読んだ条文」を画面に出しているのはなぜでしょうか
- このボットの答えだけを信じて行動してよいでしょうか